# Notebook Configuration & Parameters


In [ ]:
dataset = None

In [ ]:
if dataset is None:
    raise ValueError("No dataset provided!")

# Essential Setup & Dependencies


In [ ]:
import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), "."))  

if project_root not in sys.path:
    sys.path.append(project_root)

print("Project Root:", project_root) 

## Importing Required Libraries


In [ ]:
import sys
import random
import logging
from pathlib import Path

import pandas as pd
from critdd import Diagram

from Utils import load_edges

In [ ]:
# Configure logging

log_filename = Path("logs") / dataset / f"critdd_{dataset}.log"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler(log_filename, mode="w"),  # Overwrite log file
        logging.StreamHandler(sys.stdout)  # Print to console
    ]
)

In [ ]:
logging.basicConfig(stream=sys.stdout, level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

## Utility Functions for Data Processing


In [ ]:
def f1_score(precision: float, recall: float)-> float:
  """
  Calculates the F1-score for given precision and recall.

  Parameters
  ----------
  precision : float
      The respective precision value.
  recall : float
      The respective recall value.

  Returns
  -------
  float
      The computed F1-score based on precision and recall.
  """
  return 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

In [ ]:
def find_best_compared_graphs(df: pd.DataFrame, metric: str) -> tuple[dict[str: list[str]], set[str]]:
    """
    Finds the best graph of the corresponding method for each consistency graph.

    Parameters
    ----------
    df : pd.DataFrame
        The dataframe containing the comparisons between method's graphs and consistency graphs.
    metric : str
        The metric to decide which graph was the best match.

    Returns
    -------
    Tuple[Dict[str, List[str]], Set[str]]
        A dictionary mapping each consistency graph to its best matches among method's graphs, and a set containing all unique 
        best-matching graphs.
    """
    # Find best matches
    best_graphs = (
        df.loc[df.groupby("Reference Graph")[metric].idxmax(), ["Reference Graph", "Compared Graph"]]
        .groupby("Reference Graph")["Compared Graph"]
        .apply(list)
        .to_dict()
    )

    # Extract unique winners
    unique_winners = set(graph for graphs in best_graphs.values() for graph in graphs)
    
    return best_graphs, unique_winners

In [ ]:
def get_highest_scores(method: pd.DataFrame, score: str = "F1_score") -> pd.DataFrame:
    """
    Retrieves the highest-scoring graphs for each reference graph.

    Parameters
    ----------
    method : pd.DataFrame
        The DataFrame containing the scores for different graphs.
    score : str, optional (default="F1_score")
        The column name representing the score metric to compare.

    Returns
    -------
    pd.DataFrame
        A DataFrame containing the best-performing graphs for each reference graph.
    """
    # Compute highest scores 
    highest_scores = (
        method.loc[method.groupby("Reference Graph")[score].idxmax()]
        .set_index("Reference Graph")
    )

    return highest_scores


# Initial Data Exploration & Overview


## Analyzing Generated Graph Structures


###  Consistency Graph Selection

In [ ]:
consistency_path = Path('GeneratedGraphs/Consistency') / dataset
consistency_file = random.choice(os.listdir(consistency_path)) 

In [ ]:
consistency_file

In [ ]:
consistency_graph = load_edges(consistency_path, consistency_file)

In [ ]:
len(consistency_graph)

### PCA-Based Graph Representation

In [ ]:
pca_path = Path('GeneratedGraphs/PCA') / dataset
pca_file = random.choice(os.listdir(pca_path)) 

In [ ]:
pca_file

In [ ]:
pca_graph = load_edges(pca_path, pca_file)

In [ ]:
len(pca_graph)

### t-SNE-Based Graph Representation

In [ ]:
tsne_path = Path('GeneratedGraphs/TSNE') / dataset
tsne_file = random.choice(os.listdir(tsne_path)) 

In [ ]:
tsne_file

In [ ]:
tsne_graph = load_edges(tsne_path, tsne_file)

In [ ]:
len(tsne_graph)

In [ ]:
tsne_pca_path = Path('GeneratedGraphs/TSNE_PCA') / dataset
tsne_pca_file = random.choice(os.listdir(tsne_pca_path)) 

In [ ]:
tsne_pca_file

In [ ]:
tsne_pca_graph = load_edges(tsne_pca_path, tsne_pca_file)

In [ ]:
len(tsne_pca_graph)

### UMAP-Based Graph Representation

In [ ]:
umap_path = Path('GeneratedGraphs/UMAP') / dataset
umap_file = random.choice(os.listdir(umap_path)) 

In [ ]:
umap_file

In [ ]:
umap_graph = load_edges(umap_path, umap_file)

In [ ]:
len(umap_graph)

# Performance Metrics & Evaluation

In [ ]:
evaluation_path = Path('EvaluationResults/') / dataset

In [ ]:

pca = pd.read_parquet(evaluation_path / 'comparison_results_PCA.parquet')
tsne = pd.read_parquet(evaluation_path / 'comparison_results_TSNE.parquet')
tsne_pca = pd.read_parquet(evaluation_path / 'comparison_results_TSNE+PCA.parquet')
umap = pd.read_parquet(evaluation_path / 'comparison_results_UMAP.parquet')

## PCA

In [ ]:
pca

In [ ]:
pca['F1_score'] = pca.apply(lambda row: f1_score(row['Precision'], row['Recall']), axis=1)
pca

In [ ]:
pca.describe()

In [ ]:
pca[pca.Precision == pca.Precision.max()]

In [ ]:
pca[pca.Recall == pca.Recall.max()]

In [ ]:
pca[pca.F1_score == pca.F1_score.max()]

In [ ]:
compared_graph_max, winners = find_best_compared_graphs(pca, 'F1_score')

In [ ]:
compared_graph_max

In [ ]:
winners

## t-SNE

In [ ]:
tsne

In [ ]:
tsne['F1_score'] = tsne.apply(lambda row: f1_score(row['Precision'], row['Recall']), axis=1)
tsne

In [ ]:
tsne.describe()

In [ ]:
tsne[tsne.Precision == tsne.Precision.max()]

In [ ]:
tsne[tsne.Recall == tsne.Recall.max()]

In [ ]:
tsne[tsne.F1_score == tsne.F1_score.max()]

In [ ]:
compared_graph_max, winners = find_best_compared_graphs(tsne, 'F1_score')

In [ ]:
compared_graph_max

In [ ]:
winners

## t-SNE + PCA

In [ ]:
tsne_pca

In [ ]:
tsne_pca['F1_score'] = tsne_pca.apply(lambda row: f1_score(row['Precision'], row['Recall']), axis=1)
tsne_pca

In [ ]:
tsne_pca.describe()

In [ ]:
tsne_pca[tsne_pca.Precision == tsne_pca.Precision.max()]

In [ ]:
tsne_pca[tsne_pca.Recall == tsne_pca.Recall.max()]

In [ ]:
tsne_pca[tsne_pca.F1_score == tsne_pca.F1_score.max()]

In [ ]:
compared_graph_max, winners = find_best_compared_graphs(tsne_pca, 'F1_score')

In [ ]:
compared_graph_max

In [ ]:
winners

## UMAP

In [ ]:
umap

In [ ]:
umap['F1_score'] = umap.apply(lambda row: f1_score(row['Precision'], row['Recall']), axis=1)
umap

In [ ]:
umap.describe()

In [ ]:
umap[umap.Precision == umap.Precision.max()]

In [ ]:
umap[umap.Recall == umap.Recall.max()]

In [ ]:
umap[umap.F1_score == umap.F1_score.max()]

In [ ]:
compared_graph_max, winners = find_best_compared_graphs(umap, 'F1_score')

In [ ]:
compared_graph_max

In [ ]:
winners

# Critical Difference Diagrams

In [ ]:
pca_highest_f1 = get_highest_scores(pca)
tsne_highest_f1 = get_highest_scores(tsne)
tsne_pca_highest_f1 = get_highest_scores(tsne_pca)
umap_highest_f1 = get_highest_scores(umap)

In [ ]:
pca_results = pca_highest_f1[['F1_score']].copy()
tsne_results = tsne_highest_f1[['F1_score']].copy()
tsne_pca_results = tsne_pca_highest_f1[['F1_score']].copy()
umap_results = umap_highest_f1[['F1_score']].copy()

In [ ]:
pca_results = pca_results.rename(columns={'F1_score':'pca'})
tsne_results = tsne_results.rename(columns={'F1_score':'tsne'})
tsne_pca_results = tsne_pca_results.rename(columns={'F1_score':'tsne+pca'})
umap_results = umap_results.rename(columns={'F1_score':'umap'})

In [ ]:
results = pd.concat((pca_results, tsne_results,tsne_pca_results, umap_results), axis=1)

In [ ]:
results

In [ ]:
diagram = Diagram( # from critdd package
    results.to_numpy(),
    treatment_names = results.columns,
    maximize_outcome = True
)

In [ ]:
diagram.average_ranks

In [ ]:
diagram.get_groups(alpha=.05, adjustment="holm")

In [ ]:
# export the diagram to a file
diagram.to_file(
    f"CritddResults/critdd_{dataset}.tex",
    alpha = .05,
    adjustment = "holm",
    reverse_x = True,
    axis_options = {"title": "critdd"},
)

In [ ]:
logging.info("critical difference diagrams generated successfully")